# 20. The stack, with all five CatBoost seeds

**One variable against ledger row 27** (`stack_logit_19_oof`, CV 0.967750): the same
logistic combiner, the same `C`, the same fold-wise protocol, the same folds. The
member set goes from nineteen to twenty-three, adding CatBoost seeds 2024, 7, 2025
and 13 from rows 28 to 31.

Row 27 is refit here from the same vectors rather than quoted, so the comparison is
paired per fold inside one run.

## What to expect, and what would be surprising

Row 27 gave the single CatBoost member **+0.3512**, more than double any other, and
paid for it by taking weight off the five near-identical target-encoded LightGBM
seeds. Four more CatBoost seeds are near-identical to each other in exactly the same
way, spanning 1.32e-05 of CV between them.

So the plain expectation is that the combiner **splits** the existing +0.3512 across
five members rather than adding four more weights of that size, and that the CV gain
is small. The seed-averaging history in this repo supports that: on the raw features
the fifth LightGBM seed bought 0.000022 and the curve had been halving.

The cell that checks this prints the **sum** of the CatBoost coefficients against row
27's single one. Splitting means the sum lands near 0.3512. A sum much larger would
mean the seeds are contributing independently, which nothing here predicts.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")


train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# Row 27's nineteen in row 27's order, then the four new CatBoost seeds.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("cat42", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
    ("cat2024", O / "catboost_te_seed2024_oof.npy",
     O / "catboost_te_seed2024_test.npy"),
    ("cat7", O / "catboost_te_seed7_oof.npy", O / "catboost_te_seed7_test.npy"),
    ("cat2025", O / "catboost_te_seed2025_oof.npy",
     O / "catboost_te_seed2025_test.npy"),
    ("cat13", O / "catboost_te_seed13_oof.npy", O / "catboost_te_seed13_test.npy"),
]
NEW = ["cat2024", "cat7", "cat2025", "cat13"]
CATS = ["cat42"] + NEW


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
KEEP19 = [i for i, n in enumerate(names) if n not in NEW]
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:10} {cv:.6f}" + ("   <- new" if n in NEW else ""))


23 members, oof (691369, 23), test (296302, 23)
member CV:


  te42       0.966782


  te2024     0.966771


  te7        0.966729


  te2025     0.966743


...

  cat42      0.966915


  cat2024    0.966928   <- new


  cat7       0.966920   <- new


  cat2025    0.966916   <- new


  cat13      0.966922   <- new


In [3]:
# The fold loop, run twice over the identical folds: once on row 27's nineteen and
# once with the four new CatBoost seeds added.
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


per19, test19, coef19 = run(KEEP19)
per23, test23, coef23 = run(list(range(len(names))))

print(f"{'':22} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}")
for lbl, p in (("19 members, row 27", per19), ("23 members, this run", per23)):
    print(f"{lbl:22} " + " ".join(f"{v:9.6f}" for v in p))
print()
print(f"19-member CV {per19.mean():.6f} +/- {per19.std():.6f}"
      f"   (row 27 recorded 0.967750 +/- 0.000431, "
      f"diff {per19.mean() - 0.967750:+.2e})")
print(f"23-member CV {per23.mean():.6f} +/- {per23.std():.6f}")


                          fold 0    fold 1    fold 2    fold 3    fold 4
19 members, row 27      0.967118  0.967904  0.968057  0.968282  0.967391
23 members, this run    0.967128  0.967916  0.968067  0.968305  0.967405

19-member CV 0.967750 +/- 0.000431   (row 27 recorded 0.967750 +/- 0.000431, diff +3.70e-07)
23-member CV 0.967764 +/- 0.000434


In [4]:
# Paired, on identical folds. The right test when two models share folds is the
# spread of the per-fold DIFFERENCES and how many folds it wins, not the fold spread,
# which is common to both and cancels.
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:38} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


base17 = np.array([roc_auc_score(y[folds == f], Poof["te42"][folds == f])
                   for f in range(5)])
d_new = paired(per23, per19, "23 vs 19 members (row 27)")
paired(per23, base17, "23-member vs te42 alone (row 17)")
print()
print("For scale, from this repo's own seed history: on the raw features the fifth")
print("LightGBM seed was worth +0.000022 and the step had been halving each time.")


23 vs 19 members (row 27)              +0.000014  sd 0.000005  5/5  t(4)=5.80
     per fold: +0.000009  +0.000011  +0.000011  +0.000023  +0.000014
23-member vs te42 alone (row 17)       +0.000982  sd 0.000062  5/5  t(4)=35.45
     per fold: +0.001018  +0.000940  +0.000894  +0.001041  +0.001015

For scale, from this repo's own seed history: on the raw features the fifth
LightGBM seed was worth +0.000022 and the step had been halving each time.


In [5]:
# Coefficients, and the question the header set up: does the combiner SPLIT the
# +0.3512 it gave the single CatBoost member in row 27, or add to it?
ROW27 = {"catboost_te": 0.3512, "neural": 0.1169, "lr003": 0.1138, "te13": 0.1015,
         "te42": 0.1007, "te2024": 0.0996, "lr005": 0.0984, "te2025": 0.0953,
         "bag42": 0.0878, "bag7": 0.0843, "bag13": 0.0804, "te7": 0.0798,
         "bag2025": 0.0626, "bag2024": 0.0172, "trees1000": 0.0039,
         "lr010": -0.0032, "trees2000": -0.0087, "trees300": -0.1451,
         "anchor": -0.3622}
print(f"{'member':10} {'mean':>9} {'sd across folds':>17}   {'row 27':>8}")
for i in np.argsort(-coef23.mean(axis=0)):
    was = ROW27.get("catboost_te" if names[i] == "cat42" else names[i])
    tag = "     new" if names[i] in NEW else (f"{was:>+8.4f}" if was is not None else "")
    print(f"{names[i]:10} {coef23[:, i].mean():>+9.4f} "
          f"{coef23[:, i].std():>17.4f}   {tag}")

cat_sum = sum(coef23[:, names.index(n)].mean() for n in CATS)
print(f"\nsum of the 5 CatBoost coefficients : {cat_sum:+.4f}")
print(f"row 27's single CatBoost member    : +0.3512")
print(f"difference                         : {cat_sum - 0.3512:+.4f}")
print("Near zero means the combiner split one member's weight across five near")
print("identical ones, which is what the seed spread of 1.32e-05 predicts.")
print(f"\nlargest fold-to-fold sd: {coef23.std(axis=0).max():.4f}")


member          mean   sd across folds     row 27
neural       +0.1181            0.0021    +0.1169
lr003        +0.1161            0.0137    +0.1138
lr005        +0.0998            0.0081    +0.0984
cat2024      +0.0939            0.0099        new
te13         +0.0919            0.0068    +0.1015
te2024       +0.0915            0.0133    +0.0996
te42         +0.0893            0.0072    +0.1007
te2025       +0.0891            0.0125    +0.0953
bag42        +0.0875            0.0112    +0.0878
bag7         +0.0817            0.0184    +0.0843
cat13        +0.0816            0.0069        new
cat2025      +0.0789            0.0033        new
bag13        +0.0784            0.0089    +0.0804
cat42        +0.0753            0.0032    +0.3512
cat7         +0.0720            0.0076        new
te7          +0.0711            0.0155    +0.0798
bag2025      +0.0590            0.0062    +0.0626
bag2024      +0.0143            0.0357    +0.0172
trees1000    +0.0074            0.0097    +0.0039


In [6]:
# Submission: the average of the five fold combiners, matching what every other
# submission in this repo does with its five fold models.
pred = test23.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))

# Row 25 was held back because its ordering matched the submitted row 24 at Spearman
# 0.9999995. The same question decides whether this one is worth a slot.
prev24 = pd.read_csv(S / "stack_logit_18.csv")
prev27 = pd.read_csv(S / "stack_oof_19.csv")
for df in (prev24, prev27):
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all()
t24 = pd.Series(pred).corr(pd.Series(prev24["addicted_label"].to_numpy()),
                           method="spearman")
t27 = pd.Series(pred).corr(pd.Series(prev27["addicted_label"].to_numpy()),
                           method="spearman")
print(f"spearman vs the submitted row 24 : {t24:.7f}")
print(f"spearman vs row 27, 19 members   : {t27:.7f}")
print("Row 25 sat at 0.9999995 against row 24 and was held back on that basis.")

out = S / "stack_oof_23.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"\nwrote {out.name}, {len(sub):,} rows, "
      f"range [{prob.min():.4f}, {prob.max():.4f}]")
print(f"ledger: CV {per23.mean():.6f} +/- {per23.std():.6f}, "
      f"vs row 27 {d_new.mean():+.6f} ({(d_new > 0).sum()}/5, "
      f"sd {d_new.std(ddof=1):.6f})")


spearman vs the submitted row 24 : 0.9993027
spearman vs row 27, 19 members   : 0.9999832
Row 25 sat at 0.9999995 against row 24 and was held back on that basis.



wrote stack_oof_23.csv, 296,302 rows, range [0.0000, 1.0000]
ledger: CV 0.967764 +/- 0.000434, vs row 27 +0.000014 (5/5, sd 0.000005)
